# CATAI — Sprite Generation Notebook

Génère tous les sprites pixel art pour CATAI via Stable Diffusion XL.

**Pré-requis :** Runtime > Modifier le type d'exécution > **GPU (T4)**

**Workflow :**
1. Cellule 1 : Monte Drive + clone repo + installation (~5 min)
2. Cellule 2 : Chargement des modèles — **1ère fois ~15 min** (téléchargement Drive), ensuite ~2 min
3. Cellule 3 : Configuration
4. Cellule 4 : Génération sprites de référence → **valide visuellement**
5. Cellule 5 : Génération batch (~1h30)
6. Cellule 6 : Sauvegarde résultats sur Drive

In [ ]:
# ── Cellule 1 : Drive + repo + installation ───────────────────────────────────
# Les modèles (~10 Go) sont mis en cache sur Drive.
# 1ère exécution : ~15 min de téléchargement.
# Sessions suivantes : ~2 min (chargement depuis Drive).

from google.colab import drive
drive.mount('/content/drive')

import os

# Cache HuggingFace → Google Drive (persiste entre sessions)
HF_CACHE = '/content/drive/MyDrive/catai_hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME']            = HF_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE
print(f'Cache modèles : {HF_CACHE}')

# Clone repo (sprites PIL de référence de pose)
if not os.path.exists('/content/CATAI'):
    !git clone --branch feature/sd-sprites --depth 1 \
        https://github.com/Gheop/CATAI-Linux /content/CATAI
else:
    print('Repo déjà cloné')

PIL_ANIM_DIR = '/content/CATAI/catai_linux/cute_orange_cat/animations'
print(f'Animations PIL : {os.listdir(PIL_ANIM_DIR)}')

# Installation (rapide si déjà fait)
!pip install -q diffusers transformers accelerate safetensors xformers controlnet-aux
!pip install -q huggingface_hub Pillow opencv-python-headless
print('✓ Cellule 1 terminée')

In [ ]:
# ── Cellule 2 : Chargement des modèles ───────────────────────────────────────
# 1ère fois : télécharge ~10 Go dans Drive (patient !)
# Sessions suivantes : charge depuis Drive en ~2 min.

import torch
from diffusers import (
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    AutoencoderKL,
)
from PIL import Image
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16
print(f'Device : {DEVICE} | GPU : {torch.cuda.get_device_name(0) if DEVICE=="cuda" else "aucun"}')

print('ControlNet...')
controlnet = ControlNetModel.from_pretrained(
    'diffusers/controlnet-canny-sdxl-1.0', torch_dtype=DTYPE)

print('VAE...')
vae = AutoencoderKL.from_pretrained(
    'madebyollin/sdxl-vae-fp16-fix', torch_dtype=DTYPE)

print('SDXL... (le plus long)')
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    controlnet=controlnet, vae=vae,
    torch_dtype=DTYPE, use_safetensors=True,
).to(DEVICE)

print('LoRA pixel art...')
pipe.load_lora_weights('nerijs/pixel-art-xl', weight_name='pixel-art-xl.safetensors')

print('IP-Adapter...')
pipe.load_ip_adapter(
    'h94/IP-Adapter', subfolder='sdxl_models',
    weight_name='ip-adapter_sdxl.bin')
pipe.set_ip_adapter_scale(0.6)

pipe.enable_model_cpu_offload()
print('✓ Modèles prêts')

In [ ]:
# ── Cellule 3 : Configuration ─────────────────────────────────────────────────
import cv2
from IPython.display import display

OUT_DIR = '/content/drive/MyDrive/catai_sprites_raw'
REF_DIR = '/content/drive/MyDrive/catai_references'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REF_DIR, exist_ok=True)

GEN_SIZE = 512
STEPS    = 30
CFG      = 7.5
NEGATIVE = (
    'blurry, antialiasing, gradient, realistic, 3d, photograph, '
    'multiple cats, text, watermark, deformed, ugly, low quality'
)

CHARACTERS = {
    'orange_tabby': 'cute orange tabby cat, striped fur, white belly, green eyes, pixel art, simple, limited palette',
    'black_cat':    'cute black cat, solid black fur, bright yellow eyes, white whiskers, pixel art, simple, limited palette',
    'grey_tuxedo':  'cute grey tuxedo cat, white chest and paws, blue eyes, pixel art, simple, limited palette',
    'brown_tabby':  'cute brown tabby cat, warm brown stripes, light muzzle, amber eyes, pixel art, simple, limited palette',
    'calico':       'cute calico cat, orange and black patches on white, golden eyes, pixel art, simple, limited palette',
    'siamese':      'cute siamese cat, cream body, dark brown face mask, blue eyes, pixel art, simple, limited palette',
}
SEEDS = {
    'orange_tabby': 42,
    'black_cat':    1337,
    'grey_tuxedo':  2024,
    'brown_tabby':  7777,
    'calico':       31415,
    'siamese':      9999,
}

ANIMATIONS = [
    ('running',           ['east','west'], 8,  'running, galloping, side view'),
    ('eating',            ['south'],       11, 'eating from bowl, crouched, front view'),
    ('drinking',          ['south'],       8,  'drinking water, crouched, front view'),
    ('angry',             ['south'],       9,  'angry hissing, arched back, front view'),
    ('waking-getting-up', ['south'],       9,  'stretching, waking up, yawning, front view'),
    ('sleeping-ball',     ['south'],       4,  'sleeping curled in a tight ball'),
    ('chasing-mouse',     ['east','west'], 6,  'sneaking low, nose to ground, side view'),
    ('playing-ball',      ['south'],       8,  'standing, playful, front view'),
    ('butterfly',         ['south'],       8,  'reaching up with one paw, standing on hind legs, front view'),
    ('scratching-tree',   ['east','west'], 6,  'scratching tree with front paws raised, side view'),
    ('peeing',            ['east','west'], 6,  'one hind leg raised high, side view'),
    ('pooping',           ['south'],       6,  'squatting, back to viewer'),
    ('jumping',           ['south'],       6,  'jumping, mid-air, front view'),
    ('climbing',          ['east','west'], 8,  'climbing over ledge, hanging by front paws, side view'),
    ('love',              ['south'],       6,  'happy loaf pose, eyes half closed, front view'),
    ('flat',              ['south'],       4,  'pancake flat on ground, ears back, front view'),
    ('grooming',          ['south'],       8,  'licking paw, rubbing face, front view'),
    ('rolling',           ['south'],       8,  'rolling on back, paws in air'),
    ('surprised',         ['east','west'], 5,  'startled leap sideways, puffed fur, side view'),
]

def make_canny(pil_img):
    arr   = np.array(pil_img.convert('RGB'))
    edges = cv2.Canny(arr, 50, 150)
    return Image.fromarray(edges).convert('RGB').resize((GEN_SIZE, GEN_SIZE), Image.NEAREST)

def generate(prompt, control_img, ref_img=None, seed=42, cn_scale=0.6):
    g  = torch.Generator(device=DEVICE).manual_seed(seed)
    kw = dict(
        prompt=prompt, negative_prompt=NEGATIVE,
        image=control_img, controlnet_conditioning_scale=cn_scale,
        num_inference_steps=STEPS, guidance_scale=CFG,
        generator=g, width=GEN_SIZE, height=GEN_SIZE,
    )
    if ref_img is not None:
        kw['ip_adapter_image'] = ref_img
    return pipe(**kw).images[0]

def load_pil_frame(anim, direction, fi):
    path = os.path.join(PIL_ANIM_DIR, anim, direction, f'frame_{fi:03d}.png')
    if os.path.exists(path):
        return Image.open(path).convert('RGBA')
    return Image.new('RGBA', (68, 68), (255, 255, 255, 255))

blank = Image.new('RGB', (GEN_SIZE, GEN_SIZE), (255, 255, 255))
total = sum(len(d)*n for _,d,n,_ in ANIMATIONS) * len(CHARACTERS)
print(f'✓ Config prête — {total} frames à générer au total')

In [ ]:
# ── Cellule 4 : Sprites de référence (idle) — VALIDE avant de continuer ──────

for char_id, char_desc in CHARACTERS.items():
    ref_path = os.path.join(REF_DIR, f'{char_id}__idle.png')
    if os.path.exists(ref_path):
        print(f'  {char_id} : déjà généré')
        display(Image.open(ref_path).resize((256, 256), Image.NEAREST))
        continue
    prompt = f'{char_desc}, sitting, facing viewer, white background, centered'
    img = generate(prompt, blank, ref_img=None, seed=SEEDS[char_id], cn_scale=0.1)
    img.save(ref_path)
    print(f'  {char_id} :')
    display(img.resize((256, 256), Image.NEAREST))

print()
print('⚠  Vérifie les 6 chats ci-dessus.')
print('   Pour modifier un chat : change son SEED ou sa description en cellule 3,')
print('   supprime son fichier dans Drive/catai_references/, puis relance cette cellule.')
print('   Quand tout est OK → lance la cellule 5.')

In [ ]:
# ── Cellule 5 : Génération batch ──────────────────────────────────────────────
# Sauvegarde directement sur Drive → résiste aux déconnexions.
# Reprend automatiquement là où ça s'est arrêté.

done = skipped = 0
print(f'Total : {total} frames\n' + '─'*50)

for char_id, char_desc in CHARACTERS.items():
    ref_img = Image.open(os.path.join(REF_DIR, f'{char_id}__idle.png'))

    for anim_name, directions, n_frames, pose_desc in ANIMATIONS:
        for direction in directions:
            for fi in range(n_frames):
                out_name = f'{char_id}__{anim_name}__{direction}__{fi:03d}.png'
                out_path = os.path.join(OUT_DIR, out_name)
                if os.path.exists(out_path):
                    skipped += 1; continue

                control = make_canny(load_pil_frame(anim_name, direction, fi))
                prompt  = (f'{char_desc}, {pose_desc}, '
                           f'frame {fi+1}/{n_frames}, white background, centered, pixel art')
                seed    = SEEDS[char_id] + hash(f'{anim_name}{direction}{fi}') % 10000

                generate(prompt, control, ref_img=ref_img, seed=seed).save(out_path)
                done += 1
                if done % 10 == 0:
                    pct = (done+skipped)/total*100
                    print(f'  [{done+skipped}/{total} — {pct:.0f}%] {char_id}/{anim_name}/{direction}/f{fi}')

print(f'\n✓ Terminé — {done} générées, {skipped} déjà existantes')
print(f'Résultats dans Drive : {OUT_DIR}')

In [ ]:
# ── Cellule 6 : Instructions post-traitement ──────────────────────────────────
print('Sprites générés dans Google Drive/catai_sprites_raw/')
print()
print('Sur ton poste :')
print('  pip install rembg pillow')
print('  # Télécharge Drive/catai_sprites_raw/ dans ~/Downloads/')
print('  python3 scripts/sd_postprocess.py ~/Downloads/catai_sprites_raw catai_linux/orange_tabby')